In [ ]:
!pip install -q pyvinecopulib==1.0.0 numpy==2.1.3 scipy==1.16.3 \
              scikit-learn==1.6.1 pandas==2.2.3 matplotlib==3.10.0 \
              seaborn==0.13.2 openpyxl==3.1.5
!git clone https://github.com/mohsenbenhassine/ls-vine.git
%cd ls-vine
import sys
sys.path.insert(0, ".")

In [ ]:
import numpy as np
import pandas as pd
import torch

from src.config import CFG
from src.train import train_lsvine
from src.vine_utils import empirical_pit, vine_metrics, kendall_matrix
from src.datasets import make_student_dvine, split
from src.benchmark import DEVICE

K_VALUES_S1 = [3, 5, 7, 9]
K_VALUES_S2 = [5, 8, 12, 16]


def run_sensitivity(d, k_values, seeds=None, cfg=CFG):
    seeds = seeds or cfg.seeds[:3]
    rows = []
    for seed in seeds:
        print(f"\n[d={d}] seed={seed}")
        X = make_student_dvine(d, 0.4, 4, 3500, seed)
        ds = split(X)

        for k in k_values:
            print(f"  k={k} ...", end=" ", flush=True)
            model, vine, _, _, _ = train_lsvine(
                ds["X_train"], ds["X_val"], k, cfg, verbose=False, seed=seed)
            if vine is None:
                print("FAIL")
                continue

            Xts_t = torch.tensor(ds["X_test"], dtype=torch.float32).to(DEVICE)
            model.eval()
            with torch.no_grad():
                Zts, Xhat = model(Xts_t)
                Uts = empirical_pit(
                    Zts.float().cpu().numpy()).astype(np.float64)
                Xhat_np = Xhat.float().cpu().numpy()

            ll, aic = vine_metrics(vine, Uts)
            tau_orig = kendall_matrix(ds["X_test"])
            tau_rec = kendall_matrix(Xhat_np)
            dep_rec = np.linalg.norm(tau_orig - tau_rec, "fro")

            rows.append({"d": d, "seed": seed, "k": k,
                         "AIC": aic, "DepRec": dep_rec})
            print(f"OK (AIC={aic:.2f})")

    return pd.DataFrame(rows)


# S1 (d=10)
df_sens_S1 = run_sensitivity(10, K_VALUES_S1, seeds=CFG.seeds[:3], cfg=CFG)
df_sens_S1.to_csv("results/sensitivity_S1.csv", index=False)
print("\n-- Mean AIC and DepRec per k (S1) --")
print(df_sens_S1.groupby("k")[["AIC", "DepRec"]]
      .agg(["mean", "std"]).round(3))

# S2 (d=20)
df_sens_S2 = run_sensitivity(20, K_VALUES_S2, seeds=CFG.seeds[:3], cfg=CFG)
df_sens_S2.to_csv("results/sensitivity_S2.csv", index=False)
print("\n-- Mean AIC and DepRec per k (S2) --")
print(df_sens_S2.groupby("k")[["AIC", "DepRec"]]
      .agg(["mean", "std"]).round(3))

print("\nSaved: results/sensitivity_S1.csv, results/sensitivity_S2.csv")